#### Parameter

In [0]:
dbutils.widgets.text("file", "")

#### Import libraries

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


#### Read Data

In [0]:
source_df = spark.read\
        .format("json")\
        .load(f"s3://purchase-orders-aws/source/{dbutils.widgets.get('file')}")

#### Column mapping for api fields

In [0]:
api_to_column = {
    "record_type": "RECORD TYPE",
    "cost_center": "COST CENTER",
    "cost_center_name": "COST CENTER NAME",
    "source_doc_type_code": "DOCUMENT TYPE CODE",
    "source_doc_type_description": "DOCUMENT TYPE DESCRIPTION",
    "source_doc_status_code": "DOCUMENT STATUS CODE",
    "source_doc_status_description": "DOCUMENT STATUS DESCRIPTION",
    "vouched_amount": "VOUCHED AMOUNT",
    "estimated_start_date": "START DATE",
    "source_document": "DOCUMENT NUMBER",
    "initial_expiration_date": "EXPIRATION DATE",
    "extended_through_date": "EXTENSION DATE",
    "annual_contract": "ANNUAL CONTRACT",
    "vendor_name_1": "VENDOR NAME 1",
    "vendor_name_2": "VENDOR NAME 2",
    "vendor_city": "VENDOR CITY",
    "vendor_state": "VENDOR STATE",
    "vendor_zip": "VENDOR ZIP",
    "source_doc_type": "SOURCE DOCUMENT TYPE",
    "vendor_type": "VENDOR TYPE",
    "vendor_gender": "GENDER",
    "vendor_ethnicity": "ETHNICITY",
    "vendor_status": "STATUS",
    "vendor_class": "CLASS",
    "vendor_geographic_code": "GEOGRAPHIC AREA",
    "vendor_independent_contractor": "INDEPENDENT CONTRACTOR",
    "source_doc_desc": "DOCUMENT DESCRIPTION",
    "isminority": "MINORITY",
    "dbe": "DISADVANTAGED",
    "dvbe": "DISABLED VETERAN",
    "dvse": "SB DISABLED VET",
    "smb_min": "SB MINORITY",
    "smb_mn_wom": "SB MINORITY WOMAN",
    "sb_non_mn": "SB NON-MINORITY",
    "smb_dbe": "SB DISADVANTAGED",
    "vet_sm_bus": "SB VETERAN",
    "smb_woman": "SB WOMAN",
    "requisition_no": "REQUISITION NUMBER",
    "total_items": "TOTAL ITEMS",
    "item_number": "ITEM NUMBER",
    "item_description": "ITEM DESCRIPTION",
    "item_quantity_ordered": "ITEM QUANTITY ORDERED",
    "item_unit_of_mea": "ITEM UNIT OF MEASURE",
    "item_unit_cost": "ITEM UNIT COST",
    "item_total_cost": "ITEM TOTAL COST",
    "commodity_code": "COMMODITY CODE",
    "commodity_desc": "COMMODITY DESCRIPTION",
    "input_date": "INPUT DATE",
    "object": "EXPENSE TYPE",
    "object_desc": "EXPENSE TYPE DESCRIPTION",
    "total_amount": "TOTAL AMOUNT",
    "dept_num": "DEPARTMENT NUMBER",
    "dept_name": "DEPARTMENT NAME"
}

renamed_df = source_df
for api_col, new_col in api_to_column.items():
    if api_col in renamed_df.columns:
        renamed_df = renamed_df.withColumnRenamed(api_col, new_col)

#### Drop unnecessary columns

In [0]:
final_df = renamed_df.drop(':created_at', ':id', ':updated_at', ':version')

#### Write

In [0]:
final_df\
    .write\
    .mode('overwrite')\
    .format("csv")\
    .option("header", "true")\
    .save("s3://purchase-orders-aws/bronze/source.csv")